# DR-VERGE — ONNX export & figure repair

Run this against an **existing** artifacts folder. It does **not** retrain anything and does not
touch any result.

| Fixes | |
|---|---|
| **Issue 1** | every ONNX export failed with `ModuleNotFoundError: No module named 'onnxscript'` |
| **Issue 3a** | `fig_08` caption says "ShiftMAE" but the panel plots **ShiftL1** |
| **Issue 3b** | `fig_11` is missing the internal DRTiD bar for `best_csd_fp32` |

INT8 models are deliberately **not** exported: eager-mode quantized modules hold packed weights
(`Conv2dPackedParamsBase`) that neither ONNX nor `torch.export` can trace. That is expected — the
INT8 deployment path is *quantized skeleton + `state_dict`*, not a traced graph.

In [ ]:
# ---- 1. the missing dependency, then locate the run --------------------------------------------
!pip install -q onnxscript onnx onnxruntime

import os, json, glob, shutil
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tv

try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

DRIVE_BASE = "/content/drive/MyDrive/DR-VERGE"

# Auto-discover the artifacts folder; set ART by hand if you have several.
_cands = sorted(glob.glob(f"{DRIVE_BASE}/artifacts_*"), key=os.path.getmtime, reverse=True)
ART = _cands[0] if _cands else None
assert ART, f"no artifacts_* folder under {DRIVE_BASE} -- set ART manually"
MODELS_DIR, FIG_DIR = f"{ART}/models", f"{ART}/results/figures"
MET_DIR, TAB_DIR    = f"{ART}/results/metrics", f"{ART}/results/tables"

print("artifacts :", ART)
print("models    :", sorted(os.listdir(MODELS_DIR)) if os.path.isdir(MODELS_DIR) else "MISSING")
print("torch     :", torch.__version__)

## Model definitions — copied verbatim from the run, so `state_dict` loads exactly

In [ ]:
# ---- 2. architectures ---------------------------------------------------------------------------
# These must match the classes that produced the checkpoints, or load_state_dict will fail on shapes.
NUM_CLASSES, IMG_SIZE = 5, 224
STUDENT_CHANNELS = (32, 64, 96, 128, 160, 192, 224)
FUSION_TYPE = "interaction_mlp"

class CORALHead(nn.Module):
    def __init__(self, in_dim, num_classes=NUM_CLASSES, init_thresholds=None):
        super().__init__()
        self.num_thresholds = num_classes - 1
        self.fc = nn.Linear(in_dim, 1, bias=False)
        if init_thresholds is None:
            init_thresholds = [-0.7 * i for i in range(self.num_thresholds)]
        t = torch.tensor(list(init_thresholds), dtype=torch.float32)
        gaps = (t[:-1] - t[1:]).clamp_min(1e-4)
        self.base_bias  = nn.Parameter(t[0].clone())
        self.bias_steps = nn.Parameter(torch.log(torch.expm1(gaps)).clone())
    def _ordered_biases(self):
        steps = F.softplus(self.bias_steps)
        cum = torch.cat([torch.zeros(1, device=steps.device), torch.cumsum(steps, dim=0)])
        return self.base_bias - cum
    def forward(self, z):
        logits = self.fc(z) + self._ordered_biases().unsqueeze(0)
        return logits, torch.sigmoid(logits)

class InteractionFusion(nn.Module):
    def __init__(self, feat_dim, fusion_type=FUSION_TYPE, hidden_dim=None):
        super().__init__()
        self.fusion_type = fusion_type
        hidden_dim = hidden_dim or feat_dim
        self.norm     = nn.LayerNorm(feat_dim * 2)
        self.norm_in  = nn.LayerNorm(feat_dim * 4)
        self.proj     = nn.Linear(feat_dim * 4, hidden_dim)
        self.act      = nn.ReLU(inplace=True)
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.out_dim  = feat_dim * 2 if fusion_type == "linear" else hidden_dim
    def forward(self, z_m, z_d):
        if self.fusion_type == "linear":
            return self.norm(torch.cat([z_m, z_d], dim=1))
        combined = self.norm_in(torch.cat([z_m, z_d, torch.abs(z_m - z_d), z_m * z_d], dim=1))
        return self.norm_out(self.act(self.proj(combined)))

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, i, o, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(i, i, 3, stride=stride, padding=1, groups=i, bias=False)
        self.bn1 = nn.BatchNorm2d(i); self.act1 = nn.ReLU(inplace=True)
        self.pw = nn.Conv2d(i, o, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(o); self.act2 = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act2(self.bn2(self.pw(self.act1(self.bn1(self.dw(x))))))

class LightweightBackbone(nn.Module):
    def __init__(self, channels=None):
        super().__init__()
        ch = tuple(channels or STUDENT_CHANNELS)
        self.stem_conv = nn.Conv2d(3, ch[0], 3, stride=2, padding=1, bias=False)
        self.stem_bn = nn.BatchNorm2d(ch[0]); self.stem_act = nn.ReLU(inplace=True)
        strides = [2 if i % 2 == 0 else 1 for i in range(len(ch) - 1)]
        self.blocks = nn.ModuleList([DepthwiseSeparableBlock(ch[i], ch[i+1], strides[i])
                                     for i in range(len(ch) - 1)])
        self.gap = nn.AdaptiveAvgPool2d(1); self.out_dim = ch[-1]
    def forward(self, x):
        x = self.stem_act(self.stem_bn(self.stem_conv(x)))
        for b in self.blocks: x = b(x)
        return self.gap(x).flatten(1)

class _DualViewBase(nn.Module):
    def forward(self, macula, disc):
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        z_f = self.fusion(z_m, z_d)
        ld, pd_ = self.main_head(z_f)
        lm, pm  = self.macula_head(z_m)
        ldd, pdd = self.disc_head(z_d)
        return {"p_dual": pd_, "logit_dual": ld, "p_macula": pm, "logit_macula": lm,
                "p_disc": pdd, "logit_disc": ldd, "z_fused": z_f}

class DualViewResNetTeacher(_DualViewBase):
    def __init__(self, num_classes=NUM_CLASSES, feat_dim=2048, fusion_type=FUSION_TYPE):
        super().__init__()
        # weights=None: the ImageNet weights are overwritten by the checkpoint anyway, and this
        # avoids a needless download.
        bb = tv.resnet50(weights=None); bb.fc = nn.Identity()
        self.backbone = bb
        self.fusion = InteractionFusion(feat_dim, fusion_type)
        self.main_head   = CORALHead(self.fusion.out_dim, num_classes)
        self.macula_head = CORALHead(feat_dim, num_classes)
        self.disc_head   = CORALHead(feat_dim, num_classes)

class DualViewLightStudent(_DualViewBase):
    def __init__(self, num_classes=NUM_CLASSES, fusion_type=FUSION_TYPE):
        super().__init__()
        self.backbone = LightweightBackbone()
        fd = self.backbone.out_dim
        self.fusion = InteractionFusion(fd, fusion_type)
        self.main_head   = CORALHead(self.fusion.out_dim, num_classes)
        self.macula_head = CORALHead(fd, num_classes)
        self.disc_head   = CORALHead(fd, num_classes)

class InferenceWrapper(nn.Module):
    """ONNX gets ONE tensor out: the 4 cumulative scores P(y>k). grade = (p > 0.5).sum(1)."""
    def __init__(self, m): super().__init__(); self.model = m
    def forward(self, macula, disc): return self.model(macula, disc)["p_dual"]

print("architectures defined")

## Issue 1 — export ONNX, then **verify parity** before trusting it

An export that runs is not an export that is correct. Each model is checked against PyTorch on the
same batch and must agree to `< 1e-4` **and** predict identical grades.

In [ ]:
# ---- 3. ONNX export + parity ---------------------------------------------------------------------
import onnxruntime as ort

TOL = 1e-4
BATCH = 8                      # parity on a BATCH: a batch-handling bug is invisible at batch 1
torch.manual_seed(0)
EX = (torch.randn(BATCH, 3, IMG_SIZE, IMG_SIZE), torch.randn(BATCH, 3, IMG_SIZE, IMG_SIZE))

# INT8 models are absent by design -- eager quantized ops cannot be traced by ONNX or torch.export.
FP32_MODELS = {"teacher_fp32": DualViewResNetTeacher,
               "best_student_fp32": DualViewLightStudent,
               "best_csd_fp32": DualViewLightStudent}

results = []
for name, cls in FP32_MODELS.items():
    d = f"{MODELS_DIR}/{name}"
    ck = f"{d}/checkpoint.pt"
    if not os.path.exists(ck):
        print(f"  [{name}] SKIP -- no checkpoint.pt"); continue
    try:
        m = cls()
        state = torch.load(ck, map_location="cpu", weights_only=False)
        if isinstance(state, dict) and "model_state" in state: state = state["model_state"]
        m.load_state_dict(state); m.eval()
        wrap = InferenceWrapper(m).eval()

        out = f"{d}/model.onnx"
        torch.onnx.export(wrap, EX, out,
                          input_names=["macula", "disc"], output_names=["p_cumulative"],
                          dynamic_axes={"macula": {0: "batch"}, "disc": {0: "batch"},
                                        "p_cumulative": {0: "batch"}},
                          dynamo=True)

        with torch.no_grad(): ref = wrap(*EX).numpy()
        sess = ort.InferenceSession(out, providers=["CPUExecutionProvider"])
        got = sess.run(None, {"macula": EX[0].numpy(), "disc": EX[1].numpy()})[0]

        diff = float(np.max(np.abs(got - ref)))
        same = bool(((got > 0.5).sum(1) == (ref > 0.5).sum(1)).all())
        finite = bool(np.isfinite(got).all())
        ok = diff < TOL and same and finite
        results.append({"model": name, "onnx": True, "max_abs_diff": diff,
                        "same_grades": same, "all_finite": finite, "parity_ok": ok,
                        "size_mb": round(os.path.getsize(out) / 1024**2, 2)})
        print(f"  [{name}] {'OK  ' if ok else 'FAIL'} max|diff|={diff:.2e} "
              f"same_grades={same} finite={finite} ({os.path.getsize(out)/1024**2:.2f} MB)")
    except Exception as e:
        results.append({"model": name, "onnx": False, "error": repr(e)})
        print(f"  [{name}] EXPORT FAILED: {e!r}")

REP = pd.DataFrame(results)
REP.to_csv(f"{TAB_DIR}/table_onnx_export_report.csv", index=False)
print()
print(REP.to_string(index=False))
_ok = REP.get("parity_ok", pd.Series(dtype=bool)).fillna(False)
print(f"\n{int(_ok.sum())}/{len(REP)} models exported AND passed parity (tolerance {TOL:g})")

## Issue 3 — repair the two figure defects

Both are presentation-only. No number changes.

In [ ]:
# ---- 4a. fig_08 caption: the panel plots ShiftL1, the caption said ShiftMAE ----------------------
# They are different quantities (ShiftMAE = ShiftL1 / n_thresholds), so the old caption mislabelled
# the axis it describes.
cap = (f"{FIG_DIR}/fig_08_csd_mechanism_caption.txt")
NEW = ("Mechanism fidelity. ShiftL1 lower = the student's decision-shift is closer to the teacher's; "
       "CosAgree higher = the shift points in the same direction; BenefitCorr higher = the student "
       "gains from dual-view input on the same eyes the teacher does. Bars are the mean across "
       "seeds, error bars the SD.")
if os.path.exists(cap):
    print("OLD:", open(cap).read().strip()[:160])
with open(cap, "w") as f: f.write(NEW)
print("NEW:", NEW[:160])

In [ ]:
# ---- 4b. fig_11: fill the missing internal bar for best_csd_fp32 --------------------------------
# best_csd_fp32 IS dual_csd at the selected CSD seed, so its internal DRTiD metrics already exist in
# all_conditions_raw.csv -- the figure simply never looked them up, and silently drew nothing.
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 400, "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

DATA = f"{FIG_DIR}/fig_11_external_generalization_data.csv"
df = pd.read_csv(DATA)
RAW = pd.read_csv(f"{MET_DIR}/all_conditions_raw.csv")

sel = json.load(open(f"{ART}/configs/model_selection.json"))
csd_seed = int(sel.get("best_csd_seed"))
print(f"best_csd_fp32 == dual_csd seed {csd_seed}")

row = RAW[(RAW.condition == "dual_csd") & (RAW.seed.astype(str) == str(csd_seed))]
assert len(row), f"no dual_csd seed {csd_seed} row in all_conditions_raw.csv"
internal = {m: float(row.iloc[0][m]) for m in ("QWK", "MacroF1", "Accuracy")}
print("recovered internal values:", {k: round(v, 4) for k, v in internal.items()})

filled = 0
for i, r in df.iterrows():
    if r["condition"] == "best_csd_fp32" and pd.isna(r["internal_DRTiD"]):
        df.at[i, "internal_DRTiD"] = internal[r["metric"]]
        df.at[i, "delta_external_minus_internal"] = r["external_DeepDRiD"] - internal[r["metric"]]
        filled += 1
df.to_csv(DATA, index=False)
print(f"filled {filled} blank internal cells")

LABEL = {"teacher": "Teacher (ResNet-50, dual-view)", "best_fp32": "M* (FP32)",
         "best_csd_fp32": "Best CSD (FP32)", "ptq_int8": "PTQ INT8", "qat_int8": "QAT INT8"}
mets = ["QWK", "MacroF1", "Accuracy"]
conds = [c for c in LABEL if c in set(df.condition)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, met in zip(axes, mets):
    sub = df[df.metric == met].set_index("condition")
    iv = [sub.loc[c, "internal_DRTiD"] for c in conds]
    ev = [sub.loc[c, "external_DeepDRiD"] for c in conds]
    x = np.arange(len(conds))
    ax.bar(x - 0.19, iv, 0.38, label="DRTiD (internal)")
    ax.bar(x + 0.19, ev, 0.38, label="DeepDRiD (external, frozen)")
    ax.set_xticks(x); ax.set_xticklabels([LABEL[c] for c in conds], rotation=30, ha="right", fontsize=8)
    ax.set_title(f"{met} (higher is better)"); ax.legend(fontsize=8)
fig.suptitle("Figure 11 - Internal vs external generalization "
             "(DeepDRiD validation partition, primary field ordering)", y=1.02)
for ext in ("png", "pdf", "svg"):
    fig.savefig(f"{FIG_DIR}/fig_11_external_generalization.{ext}", bbox_inches="tight")
plt.close(fig)

with open(f"{FIG_DIR}/fig_11_external_generalization_caption.txt", "w") as f:
    f.write("Internal (DRTiD test) versus external (DeepDRiD, frozen) performance. A drop on the "
            "external set is a domain-shift finding to report, not something to tune away.")
print("fig_11 re-rendered with every internal bar present")

In [ ]:
# ---- 5. summary ---------------------------------------------------------------------------------
print("=" * 78)
print("REPAIR SUMMARY")
print("=" * 78)
onnx_files = sorted(glob.glob(f"{MODELS_DIR}/*/model.onnx"))
print(f"  ONNX exported : {len(onnx_files)}")
for p in onnx_files:
    print(f"      {os.path.basename(os.path.dirname(p)):24} {os.path.getsize(p)/1024**2:6.2f} MB")
print(f"  parity report : {TAB_DIR}/table_onnx_export_report.csv")
blank = pd.read_csv(DATA)["internal_DRTiD"].isna().sum()
print(f"  fig_11 blanks : {blank} (0 = fixed)")
print(f"  fig_08 caption: now names ShiftL1, matching the panel")
print()
print("  NOT exported, by design: best_student_ptq_int8 / best_student_qat_int8 --")
print("  eager quantized modules cannot be traced by ONNX. Serve those from")
print("  quantized skeleton + checkpoint.pt in a Python backend (see knowledge/).")